# 1. Generating Atmospheric Turbulence with `PhaseDataset`

Part 1 of a four-notebook tutorial series — `01_Dataset.ipynb` → `02_WFSAndPreprocessing.ipynb` → `03_DeformableMirrorAndClosedLoop.ipynb` → `04_TrainingAReconstructor.ipynb` — that builds AI4AO's simulation pipeline one stage at a time, ending with a trained phase reconstructor. Every notebook in the series configures the same synthetic, uncalibrated instrument from `wfs_params_exp.py`; there's no real optical bench behind it, which is what makes it a good place to learn the pipeline without needing bench data.

This notebook covers `AI4AO.PhaseDataset`, the source of every ground-truth phase screen used later in the series. It generates atmospheric turbulence on the fly from a von Kármán power spectral density (PSD) — there's no WFS, DM, or network yet, just the physics of the turbulence itself.

## Configuration

`wfs_params_exp.py` defines plain Python dicts — `WFSParams`, `AtmosParams`, `LoopParams`, `DMParams`, `TrainParams` — consumed positionally by the pipeline constructors. This is the configuration mechanism throughout AI4AO; there is no YAML/JSON config layer. `PhaseDataset` only needs `WFSParams`, `AtmosParams`, `LoopParams` and `DMParams` (not `TrainParams`, which is for the optimizer in notebook 4).

### What's inside `wfs_params_exp.py`

The five dicts cover every stage of the pipeline; `PhaseDataset` alone reads from all of them except `TrainParams`. The fields it actually uses here:

- **`WFSParams`**: `Nres` (pupil sampling, in pixels across the telescope diameter), `D` (telescope diameter, m), `Wavelength` (sensing wavelength, m), `Nphotons`/`RON` (log-range of photons per frame, and read-out-noise range in electrons/pixel/frame — drawn per sample into the batch's `"nphotons"`/`"ron"`, used from notebook 2 onward once there's a detector to add noise to). `sampling`, `centralObstruction`, `useNoise`, `Modulation` etc. also live in `WFSParams` but only matter once there's a WFS mask (notebook 2).
- **`AtmosParams`**: `r0` (Fried parameter range, m — smaller means stronger turbulence), `L0` (outer-scale range, m), `Nphases` (number of independent phase screens per batch), `Layers` (range for the random number of turbulent layers per sample), `f_slope` (PSD power-law slope, `11/6` is the standard Kolmogorov/von Kármán value), `Scintillation` (whether to also propagate amplitude fluctuations, see below).
- **`LoopParams`**: describes the AO loop assumed when `generateClosedLoop = True` — `loopFrequency` (Hz), `delayFrames` (frames of pure delay), `windSpeedVector` (range, m/s, used to translate layers between samples), `levelOfCorrection` (range, how much of the DM-correctable spatial frequencies are actually removed from the residual PSD), `loopGain`/`loopLeak` (range for the leaky-integrator controller — drawn per sample into `"loop_gain"`/`"loop_leak"`, used again in notebook 3).
- **`DMParams`**: only `Nactuator` is read here, to set the spatial-frequency cutoff of the DM-correctable PSD used for closed-loop residuals. The rest of `DMParams` (`Nmodes`, `moffatParam`, `signedAmplitude`, `MechCoupling`, `FlipLeftRight`, `FlipTopBottom`) configures the physical `DeformableMirror` object introduced in notebook 3.

`TrainParams` (optimizer learning rates, number of training/test steps, `OptimizeMask`) isn't touched until notebook 4.

In [1]:
from mmengine import Config
import matplotlib.pyplot as plt

from AI4AO import PhaseDataset

device = 'cuda'  # set to "cpu" if CUDA is not available

paramfile = 'wfs_params_exp.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']

## Creating the dataset

`PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)` is `Dataset`-like: indexing it with `dataset[idx]` returns a batch dict of tensors (see the last section of this notebook for the full list of keys).

In [ ]:
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)

## Open-loop phase screens

`dataset.generateClosedLoop` starts out `False`. In this mode `dataset[idx]` draws phase screens from the raw atmospheric PSD — uncorrected turbulence, with no AO loop assumed. Indexing always advances the *same* turbulence realization by `idx` wind-shifted time steps; a brand-new independent realization is only drawn when you index `dataset[0]` again. So below: both `dataset[0]` calls start a fresh realization each time, `dataset[1]` advances the second one by one time step, and `dataset[5]` advances it to `t = 5T`.

In [ ]:
plt.figure(figsize=(12, 3))

batch = dataset[0]
plt.subplot(141)
plt.title('New phase screen')
plt.imshow(batch["phase"][0].cpu())

batch = dataset[0]  # dataset[0] always draws a fresh realization
plt.subplot(142)
plt.title('New phase screen')
plt.imshow(batch["phase"][0].cpu())

batch = dataset[1]  # advances the same realization by one time step
plt.subplot(143)
plt.title('Same screen at t = T')
plt.imshow(batch["phase"][0].cpu())

batch = dataset[5]  # ...and to t = 5T
plt.subplot(144)
plt.title('Same screen at t = 5T')
plt.imshow(batch["phase"][0].cpu())
plt.show()

Stepping through consecutive indices animates the turbulence being blown across the pupil by the per-layer wind. The quiver arrows below show each layer's wind vector, scaled/faded by that layer's fractional contribution to `r0` (`fractional_r0`) — the dominant layer's arrow is the most visible. The wind vectors themselves are fixed for the whole realization (drawn once, when `dataset[0]` is indexed), so only the phase images need to be redrawn per frame.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_frames = 60

batch0 = dataset[0]
wind = batch0["wind"][..., :4].cpu()
fractional_r0 = batch0["fractional_r0"][:, :4].cpu()

phases = [batch0["phase"][:4].cpu()]
for i in range(1, n_frames):
    phases.append(dataset[i]["phase"][:4].cpu())

frame_center = phases[0].shape[-1] // 2
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
images = []
for j in range(4):
    images.append(axes[j].imshow(phases[0][j]))
    axes[j].set_title(f"Phase {j + 1}")
    axes[j].axis('off')

    x = [frame_center] * wind.shape[1]
    y = [frame_center] * wind.shape[1]
    u, v = wind[1, :, j], wind[0, :, j]
    alpha = (fractional_r0[:, j] / fractional_r0[:, j].max()).numpy()
    axes[j].quiver(x, y, u, v, angles="xy", scale_units="xy", scale=0.3, alpha=alpha)


def update(i):
    for j in range(4):
        images[j].set_data(phases[i][j])
    return images


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())

## Closed-loop (AO-residual) phase screens

Setting `dataset.generateClosedLoop = True` does **not** change how indexing evolves in time — that behavior is unaffected by this flag, as shown above. What it changes is the PSD the phase screens are drawn from: it blends in a spatial "fitting" PSD (the spatial-frequency content a DM with `DMParams['Nactuator']` actuators can correct, scaled by `LoopParams['levelOfCorrection']`) and a temporal-error PSD (from `loopFrequency`, `loopGain`, `loopLeak`, `delayFrames`), so the phase already looks like the residual downstream of a running AO loop, rather than raw open-loop turbulence. This is the more realistic target for training a reconstructor, and what notebooks 3 and 4 use.

Compare the phase screens' spatial structure below to the open-loop ones above — the AO-residual phase should look smoother/more band-limited, since the low spatial frequencies a DM can correct are suppressed.

In [ ]:
dataset.generateClosedLoop = True

plt.figure(figsize=(12, 3))

batch = dataset[0]
plt.subplot(141)
plt.title('New phase screen')
plt.imshow(batch["phase"][0].cpu())

batch = dataset[0]
plt.subplot(142)
plt.title('New phase screen')
plt.imshow(batch["phase"][0].cpu())

batch = dataset[1]
plt.subplot(143)
plt.title('Same screen at t = T')
plt.imshow(batch["phase"][0].cpu())

batch = dataset[5]
plt.subplot(144)
plt.title('Same screen at t = 5T')
plt.imshow(batch["phase"][0].cpu())
plt.show()

## Multi-layer turbulence and the Cn² profile

Every time you index `dataset[0]`, `DrawRandomParameters` draws a random number of layers, each with a random height from an exponenitial distribution and a random fractional contribution to `r0` (`dataset.fractionalr0`) — i.e. a normalized `Cn²` profile. One curve per phase in the batch below, since these are drawn independently per sample. You can change the values of `dataset.height_exp_dist_lambda` which controlls the height distribution

In [ ]:
dataset.height_exp_dist_lambda = 0.0003
batch = dataset[0]

for index in range(dataset.layerHeights.shape[1]):
    plt.semilogx(
        dataset.layerHeights.cpu()[:, index].squeeze(),
        dataset.fractionalr0.cpu()[:, index].squeeze(),
        'o-'
    )

plt.xlabel('Height (m)')
plt.ylabel('Relative $C_n^2$')
plt.show()

## Scintillation

`AtmosParams['Scintillation'] = True` makes `PhaseDataset` additionally propagate each turbulence layer through free space with an angular-spectrum propagator (`dataset.ASP`), so the pupil sees amplitude fluctuations (intensity scintillation) on top of the phase, not just a flat-amplitude wavefront. This requires rebuilding the dataset, since it changes what `__getitem__` computes internally.

In [ ]:
AtmosParams['Scintillation'] = True
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)

batch = dataset[0]
plt.figure(figsize=(10, 5))
plt.subplot(121)
plt.imshow((batch["pupil"][0] * dataset.pupil).cpu())
plt.title('Pupil amplitude')
plt.axis('off')
plt.colorbar()
plt.subplot(122)
plt.imshow(batch["phase"][0].cpu())
plt.title('Phase')
plt.axis('off')
plt.colorbar()
plt.show()

### The scintillation's full support

`ComputeScintillation` returns a complex field over a padded support larger than the telescope pupil — needed so the angular-spectrum propagation doesn't wrap/alias at the edges. The red circle below marks the actual telescope pupil (`dataset.Nres` wide) within that padded support.

In [ ]:
from matplotlib.patches import Circle

phasor = dataset.ComputeScintillation(dataset.layeredPhase)
support = phasor.abs()

plt.imshow(support[0].cpu())
circle = Circle(
    (support.shape[-2] / 2, support.shape[-1] / 2), dataset.Nres / 2,
    fill=False, color='red', linewidth=2
)
plt.gca().add_patch(circle)
plt.colorbar()
plt.show()

## What's in a batch

Beyond `"phase"` and `"pupil"`, every batch carries the randomly-drawn simulation parameters for that sample: `"nphotons"`/`"ron"` (WFS noise — used from notebook 2 onward), `"r0"`, `"wind"`, `"fractional_r0"`, and `"loop_gain"`/`"loop_leak"` (the closed-loop correction parameters used from notebook 3 onward).

In [ ]:
batch.keys()